# Phase 5 — ML Prediction Models
### Notebook: `phase_5_ml.ipynb`

**Two separate tasks in this notebook:**

**Task B (done first — simpler, more publishable result):**  
Correlation and linear regression: does state-level pollution predict deaths from disease?  
Input: `combined_master.csv` (state × year)  

**Task A (done second — time-series ML):**  
Predict next month's PM2.5 for a city from its current pollution and weather.  
Input: `air_pollution_with_aqi.csv` (city × month)  

---
**Model progression (never skip ahead):**  
1. Linear Regression — always the baseline. If R² ≥ 0.6, stop here.  
2. Random Forest — only if Linear R² < 0.6 (Task A only).  
3. SARIMA — optional, only for tier_1 cities with 8+ years of data.  

---
**Non-negotiable rule for Task A:**  
The train/test split is done by TIME — train on 2015–2020, test on 2021–2023.  
NEVER use `sklearn`'s `train_test_split()` on time-series data — that shuffles rows  
randomly and would let the model train on 2022 data while testing on 2019 data.  
That produces fraudulent accuracy numbers.

---
## Cell 1 — Install and Import Libraries

`scikit-learn` and `scipy` may not be in your environment yet.  
The first block installs them if missing. Run it once — it is safe to re-run.

In [1]:
# ── Install required libraries if not already present ─────────────────────────
# WHY: scikit-learn and scipy are not part of the base Anaconda install.
# This installs them into the active pollution_env environment.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'scikit-learn', 'scipy', '--quiet'], check=True)

# ── Core libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

# ── scikit-learn: ML models and evaluation metrics ────────────────────────────
from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from sklearn.metrics         import r2_score, mean_absolute_error
from sklearn.preprocessing   import StandardScaler

# ── scipy: Pearson correlation with p-value ───────────────────────────────────
# WHY scipy and not pandas .corr(): scipy gives us the p-value, which tells us
# whether the correlation is statistically significant or could be due to chance.
from scipy.stats import pearsonr

# Suppress minor sklearn warnings that don't affect results
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")

All libraries imported successfully


---
## Cell 2 — Set Paths and Load Data

In [2]:
BASE_DIR   = Path('..')
OUTPUT_DIR = BASE_DIR / 'outputs'

COMBINED_PATH  = OUTPUT_DIR / 'combined_master.csv'
POLLUTION_PATH = OUTPUT_DIR / 'air_pollution_with_aqi.csv'

for fpath in [COMBINED_PATH, POLLUTION_PATH]:
    if not fpath.exists():
        raise FileNotFoundError(
            f"File not found: {fpath}\n"
            "Run Phases 1-4 first to produce these files."
        )

df_combined  = pd.read_csv(COMBINED_PATH,  low_memory=False)
df_pollution = pd.read_csv(POLLUTION_PATH, low_memory=False)

print(f"combined_master.csv      : {df_combined.shape[0]:,} rows x {df_combined.shape[1]} columns")
print(f"air_pollution_with_aqi   : {df_pollution.shape[0]:,} rows x {df_pollution.shape[1]} columns")
print()

# ── Confirm key columns exist before we use them ─────────────────────────────
# WHY: If a Phase 3 or 4 column is missing, we get a confusing KeyError later.
# Better to check now and print a clear message.
REQUIRED_COMBINED = ['location_name', 'year', 'cause_name', 'metric_id',
                     'val', 'join_quality', 'PM2.5 (ug/m3)_mean',
                     'AQI_annual_mean', 'pct_months_Worst', 'evidence_strength']
REQUIRED_POLLUTION = ['city', 'state', 'year', 'month', 'data_quality_tier',
                      'PM2.5 (ug/m3)_mean', 'AT (degree C)_mean',
                      'RH (%)_mean', 'WS (m/s)_mean', 'RF (mm)_mean', 'AQI']

for col in REQUIRED_COMBINED:
    if col not in df_combined.columns:
        print(f"WARNING: '{col}' missing from combined_master.csv")

for col in REQUIRED_POLLUTION:
    if col not in df_pollution.columns:
        print(f"WARNING: '{col}' missing from air_pollution_with_aqi.csv")

print("Column check complete")

combined_master.csv      : 18,228 rows x 49 columns
air_pollution_with_aqi   : 10,295 rows x 102 columns

Column check complete


---
# TASK B — Does Pollution Predict Disease Deaths?

## Cell 3 — Prepare Task B Data

**Why we use Rate (metric_id = 3) not Number (metric_id = 1) for this analysis:**  
Rate = deaths per 100,000 population. This controls for state population size.  
Uttar Pradesh (200M people) will always have more raw deaths than Sikkim (700k people)  
regardless of pollution — that's not a pollution effect, that's a population effect.  
Using Rate lets us fairly compare a large state against a small state.

**Why we filter to `join_quality == 'full'`:**  
Only rows where both pollution AND disease data are present are usable for regression.  
`disease_only` rows (e.g. Goa) have no pollution columns — cannot be used as features.

In [3]:
# ── Filter to usable rows ─────────────────────────────────────────────────────
# metric_id == 3  : Rate (deaths per 100,000 population)
# join_quality == 'full' : has both pollution data and disease data
df_taskb = df_combined[
    (df_combined['metric_id']    == 3) &
    (df_combined['join_quality'] == 'full')
].copy()

print(f"Rows after filtering: {len(df_taskb):,}")
print(f"States included     : {df_taskb['location_name'].nunique()}")
print(f"Years included      : {sorted(df_taskb['year'].unique())}")
print(f"Disease causes      : {df_taskb['cause_name'].nunique()}")
print()

# ── Define pollution feature columns for Task B ───────────────────────────────
# WHY these specific columns:
# PM2.5 = the single most important health pollutant (WHO evidence)
# AQI_annual_mean = captures combined effect of all 7 pollutants
# pct_months_Worst = captures how often the state was in extreme pollution
# These three give a complete picture without overfitting (only 3 features for
# a dataset of ~300-400 state-year rows)
TASKB_FEATURES = [
    'PM2.5 (ug/m3)_mean',
    'AQI_annual_mean',
    'pct_months_Worst',
]

# ── Check for NaN in feature columns ─────────────────────────────────────────
for col in TASKB_FEATURES:
    if col in df_taskb.columns:
        n_nan = df_taskb[col].isna().sum()
        pct   = n_nan / len(df_taskb) * 100
        print(f"   {col:35s}: {n_nan:,} NaN ({pct:.1f}%)")

print()
print("Note: NaN in feature columns = that state-year had insufficient pollution data.")
print("These rows will be dropped per cause when fitting the model.")

Rows after filtering: 2,016
States included     : 17
Years included      : [np.int64(2013), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Disease causes      : 21

   PM2.5 (ug/m3)_mean                 : 0 NaN (0.0%)
   AQI_annual_mean                    : 0 NaN (0.0%)
   pct_months_Worst                   : 0 NaN (0.0%)

Note: NaN in feature columns = that state-year had insufficient pollution data.
These rows will be dropped per cause when fitting the model.


---
## Cell 4 — Task B: Pearson Correlation Per Disease Cause

The Pearson correlation coefficient (r) measures the strength of the linear relationship  
between PM2.5 and deaths from each disease.  

- r close to +1 = strong positive link (more pollution → more deaths)
- r close to 0 = no linear relationship
- p-value < 0.05 = the relationship is statistically significant (not due to chance)

This is the most directly reportable result in the project.

In [4]:
print("=" * 70)
print("TASK B — PEARSON CORRELATION: PM2.5 vs Disease Death Rate")
print("=" * 70)
print()
print(f"{'Disease Cause':<50} {'r':>6} {'p-value':>10} {'n':>6} {'Sig?':>6} {'Evidence'}")
print("-" * 110)

# Store results for saving
taskb_corr_results = []

causes = sorted(df_taskb['cause_name'].unique())

for cause in causes:
    # Filter to this cause only
    df_cause = df_taskb[df_taskb['cause_name'] == cause].copy()

    # Drop rows where PM2.5 or val is NaN — pearsonr cannot handle NaN
    df_cause = df_cause.dropna(subset=['PM2.5 (ug/m3)_mean', 'val'])

    # Need at least 10 data points for correlation to be meaningful
    if len(df_cause) < 10:
        continue

    # Compute Pearson r and p-value
    # WHY pearsonr not df.corr(): pearsonr gives us the p-value which tells us
    # whether the correlation is statistically significant
    r, p = pearsonr(df_cause['PM2.5 (ug/m3)_mean'], df_cause['val'])

    # Significance marker
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))

    # Get evidence strength for this cause
    evidence = df_cause['evidence_strength'].iloc[0]

    # Print result
    cause_short = cause[:48]
    print(f"  {cause_short:<50} {r:>6.3f} {p:>10.4f} {len(df_cause):>6} {sig:>6}   {evidence}")

    taskb_corr_results.append({
        'cause_name'       : cause,
        'pearson_r'        : round(r, 4),
        'p_value'          : round(p, 6),
        'n_observations'   : len(df_cause),
        'significant_05'   : p < 0.05,
        'evidence_strength': evidence,
    })

print()
print("Significance: *** p<0.001   ** p<0.01   * p<0.05   n.s. = not significant")
print()

# Summary
df_corr = pd.DataFrame(taskb_corr_results)
n_sig = df_corr['significant_05'].sum()
print(f"Significant correlations (p < 0.05): {n_sig} of {len(df_corr)} causes")

TASK B — PEARSON CORRELATION: PM2.5 vs Disease Death Rate

Disease Cause                                           r    p-value      n   Sig? Evidence
--------------------------------------------------------------------------------------------------------------
  Asthma                                              0.362     0.0003     96    ***   direct
  Atrial fibrillation and flutter                    -0.577     0.0000     96    ***   direct
  Cardiovascular diseases                            -0.555     0.0000     96    ***   direct
  Chronic obstructive pulmonary disease               0.182     0.0761     96   n.s.   direct
  Chronic respiratory diseases                        0.192     0.0603     96   n.s.   direct
  Congenital birth defects                            0.373     0.0002     96    ***   indirect
  Hypertensive heart disease                         -0.556     0.0000     96    ***   direct
  Interstitial lung disease and pulmonary sarcoido   -0.330     0.0010     96 

---
## Cell 5 — Task B: Linear Regression Per Disease Cause

We now fit a linear regression model for each disease cause.  
Features: PM2.5 annual mean + AQI annual mean + pct months in Worst tier  
Target: death rate (per 100k population)

**Train/test split for Task B:**  
Train on years 2010–2019 (10 years), test on years 2020–2023 (4 years).  
This is the correct temporal split — we never train on future data to predict the past.

In [5]:
print("=" * 70)
print("TASK B — LINEAR REGRESSION: Pollution Features → Death Rate")
print("=" * 70)
print()
print(f"Train years: 2010–2019  |  Test years: 2020–2023")
print(f"Features: {TASKB_FEATURES}")
print()
print(f"{'Disease Cause':<50} {'Train R²':>9} {'Test R²':>8} {'Test MAE':>10} {'n_train':>8} {'n_test':>7}")
print("-" * 100)

taskb_reg_results = []

for cause in causes:
    df_cause = df_taskb[df_taskb['cause_name'] == cause].copy()

    # Drop rows where ANY feature or target is NaN
    df_cause = df_cause.dropna(subset=TASKB_FEATURES + ['val'])

    if len(df_cause) < 15:
        # Not enough data for a meaningful train/test split
        continue

    # ── Temporal train/test split ─────────────────────────────────────────────
    # WHY: We split by year, not randomly. Predicting 2020-2023 from 2010-2019
    # mirrors the real use case: using historical data to predict the future.
    train = df_cause[df_cause['year'] <= 2019]
    test  = df_cause[df_cause['year'] >= 2020]

    if len(train) < 10 or len(test) < 5:
        continue

    X_train = train[TASKB_FEATURES].values
    y_train = train['val'].values
    X_test  = test[TASKB_FEATURES].values
    y_test  = test['val'].values

    # ── Scale features ────────────────────────────────────────────────────────
    # WHY: PM2.5 values (0-250) and pct_months_Worst (0-100) are on different
    # scales. StandardScaler brings them to mean=0, std=1 so the linear model
    # treats them equally. Fit scaler ONLY on training data — not on test data.
    # Fitting on test data would leak information about the test set.
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)   # fit + transform on train
    X_test  = scaler.transform(X_test)        # transform only on test (no fit)

    # ── Fit Linear Regression ─────────────────────────────────────────────────
    model = LinearRegression()
    model.fit(X_train, y_train)

    # ── Evaluate ──────────────────────────────────────────────────────────────
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    train_r2  = r2_score(y_train, y_pred_train)
    test_r2   = r2_score(y_test,  y_pred_test)
    test_mae  = mean_absolute_error(y_test, y_pred_test)

    cause_short = cause[:48]
    print(f"  {cause_short:<50} {train_r2:>9.3f} {test_r2:>8.3f} {test_mae:>10.3f} {len(train):>8} {len(test):>7}")

    taskb_reg_results.append({
        'cause_name'   : cause,
        'train_r2'     : round(train_r2, 4),
        'test_r2'      : round(test_r2,  4),
        'test_mae'     : round(test_mae, 4),
        'n_train'      : len(train),
        'n_test'       : len(test),
        'coefficients' : dict(zip(TASKB_FEATURES, model.coef_.round(4)))
    })

print()
print("Interpretation guide:")
print("  Test R² > 0.5  — model explains >50% of variance — strong result")
print("  Test R² 0.3-0.5 — moderate result — mention limitations")
print("  Test R² < 0.3  — weak predictive power — report correlation only")
print("  Negative test R² — model performs worse than just predicting the mean")

TASK B — LINEAR REGRESSION: Pollution Features → Death Rate

Train years: 2010–2019  |  Test years: 2020–2023
Features: ['PM2.5 (ug/m3)_mean', 'AQI_annual_mean', 'pct_months_Worst']

Disease Cause                                       Train R²  Test R²   Test MAE  n_train  n_test
----------------------------------------------------------------------------------------------------
  Asthma                                                 0.386   -0.177      2.501       37      59
  Atrial fibrillation and flutter                        0.552    0.139      0.660       37      59
  Cardiovascular diseases                                0.446    0.291     44.805       37      59
  Chronic obstructive pulmonary disease                  0.186   -0.110     13.872       37      59
  Chronic respiratory diseases                           0.207   -0.124     17.358       37      59
  Congenital birth defects                               0.187    0.073      2.249       37      59
  Hypertensive hea

---
## Cell 6 — Task B: Save Results

Save both the correlation results and the regression results to CSV  
so they can be referenced in the project report.

In [6]:
# ── Save correlation results ──────────────────────────────────────────────────
df_corr_out = pd.DataFrame(taskb_corr_results).sort_values('pearson_r', ascending=False)
corr_path   = OUTPUT_DIR / 'taskb_correlation_results.csv'
df_corr_out.to_csv(corr_path, index=False)
print(f"Correlation results saved: {corr_path}")
print(f"   {len(df_corr_out)} causes, sorted by Pearson r (strongest first)")
print()

# ── Save regression results ───────────────────────────────────────────────────
# Flatten the coefficients dict into separate columns for readability
df_reg_out = pd.DataFrame(taskb_reg_results)
if 'coefficients' in df_reg_out.columns:
    coef_df = df_reg_out['coefficients'].apply(pd.Series)
    df_reg_out = pd.concat(
        [df_reg_out.drop(columns=['coefficients']), coef_df],
        axis=1
    )
df_reg_out = df_reg_out.sort_values('test_r2', ascending=False)

reg_path = OUTPUT_DIR / 'taskb_regression_results.csv'
df_reg_out.to_csv(reg_path, index=False)
print(f"Regression results saved: {reg_path}")
print(f"   {len(df_reg_out)} causes, sorted by test R² (best first)")
print()

# ── Print top findings ────────────────────────────────────────────────────────
print("Top 5 strongest correlations (PM2.5 vs death rate):")
print(df_corr_out[['cause_name', 'pearson_r', 'p_value', 'significant_05', 'evidence_strength']]
      .head(5).to_string(index=False))

Correlation results saved: ..\outputs\taskb_correlation_results.csv
   21 causes, sorted by Pearson r (strongest first)

Regression results saved: ..\outputs\taskb_regression_results.csv
   21 causes, sorted by test R² (best first)

Top 5 strongest correlations (PM2.5 vs death rate):
                                              cause_name  pearson_r  p_value  significant_05 evidence_strength
                            Upper respiratory infections     0.6842 0.000000            True            strong
Neonatal encephalopathy due to birth asphyxia and trauma     0.6644 0.000000            True          indirect
                                  Neonatal preterm birth     0.4978 0.000000            True          indirect
                                            Otitis media     0.4360 0.000009            True            strong
                            Lower respiratory infections     0.4042 0.000044            True            strong


---
# TASK A — Predict Next Month's PM2.5

## Cell 7 — Prepare Task A Data: Filter, Create Lag Features

**What lag features are:**  
Last month's PM2.5 is one of the best predictors of this month's PM2.5 (air pollution persists).  
A "lag feature" is simply the previous row's value shifted forward by one month within each city.

**Why month needs cyclical encoding:**  
Month is a circular variable — December (12) and January (1) are one month apart,  
not 11 months apart. If we pass month as a raw number (1-12), the model thinks  
December and January are far apart. Sine/cosine encoding fixes this.

In [7]:
# ── Filter to tier_1 and tier_2 cities only ───────────────────────────────────
# WHY: Tier_3 and tier_4 cities have < 5 years of data — not enough history
# to train a time-series model and still have test years left over.
df_taska = df_pollution[
    df_pollution['data_quality_tier'].isin(['tier_1', 'tier_2'])
].copy()

print(f"Rows after tier filter: {len(df_taska):,}")
print(f"Cities: {df_taska['city'].nunique()}")
print(f"States: {df_taska['state'].nunique()}")
print(f"Year range: {df_taska['year'].min()} to {df_taska['year'].max()}")
print()

# ── Sort by city, year, month — CRITICAL before creating lag features ─────────
# WHY: shift(1) takes the PREVIOUS row. If rows are not sorted chronologically
# within each city, shift(1) will take the wrong row — silent data corruption.
df_taska = df_taska.sort_values(['city', 'year', 'month']).reset_index(drop=True)

# ── Create lag features WITHIN each city group ───────────────────────────────
# WHY groupby: without groupby, shift(1) at the start of city B would take
# the last row of city A — that is a different city, wrong data entirely.
# groupby('city').shift(1) ensures each city's first row gets NaN, not another city's data.
df_taska['PM25_lag1'] = df_taska.groupby('city')['PM2.5 (ug/m3)_mean'].shift(1)
df_taska['PM25_lag2'] = df_taska.groupby('city')['PM2.5 (ug/m3)_mean'].shift(2)

# ── Create target: NEXT month's PM2.5 ────────────────────────────────────────
# shift(-1) = next row's value. Again done within each city group.
df_taska['PM25_next'] = df_taska.groupby('city')['PM2.5 (ug/m3)_mean'].shift(-1)

# ── Cyclical month encoding ───────────────────────────────────────────────────
# WHY: Month 1 (Jan) and month 12 (Dec) are adjacent in the real world,
# but numerically they are 11 apart. Sine/cosine maps month onto a circle
# so that Dec and Jan are close together as they should be.
df_taska['month_sin'] = np.sin(2 * np.pi * df_taska['month'] / 12)
df_taska['month_cos'] = np.cos(2 * np.pi * df_taska['month'] / 12)

# ── Define feature columns for Task A ────────────────────────────────────────
TASKA_FEATURES = [
    'PM25_lag1',            # Previous month's PM2.5 — strongest predictor
    'PM25_lag2',            # Two months ago — captures persistence
    'AT (degree C)_mean',   # Temperature affects PM2.5 (inversion layers in winter)
    'RH (%)_mean',          # Humidity affects PM2.5 (wet deposition reduces it)
    'WS (m/s)_mean',        # Wind speed disperses pollution
    'RF (mm)_mean',         # Rainfall washes out particulates
    'month_sin',            # Cyclical month encoding
    'month_cos',
    'year',                 # Captures long-term trend (improving or worsening)
]

# Target column
TASKA_TARGET = 'PM25_next'

# ── Drop rows where target or lag features are NaN ───────────────────────────
# WHY: The first 2 rows of each city have NaN lags (no previous month).
# The last row of each city has NaN target (no next month).
# These rows cannot be used for training or testing.
cols_needed = TASKA_FEATURES + [TASKA_TARGET, 'year', 'city', 'state']
df_taska = df_taska.dropna(subset=cols_needed).copy()

print(f"Rows after dropping NaN lag/target rows: {len(df_taska):,}")
print(f"(Dropped first 2 + last 1 rows per city — expected and correct)")
print()
print(f"Feature columns for Task A: {TASKA_FEATURES}")
print(f"Target column: {TASKA_TARGET}")

Rows after tier filter: 7,535
Cities: 105
States: 18
Year range: 2010 to 2023

Rows after dropping NaN lag/target rows: 2,405
(Dropped first 2 + last 1 rows per city — expected and correct)

Feature columns for Task A: ['PM25_lag1', 'PM25_lag2', 'AT (degree C)_mean', 'RH (%)_mean', 'WS (m/s)_mean', 'RF (mm)_mean', 'month_sin', 'month_cos', 'year']
Target column: PM25_next


---
## Cell 8 — Task A: Temporal Train/Test Split

Train on 2015–2020 data. Test on 2021–2023 data.  
This mirrors reality: train on everything we know, predict what comes next.

In [8]:
# ── Temporal split ────────────────────────────────────────────────────────────
# WHY NOT sklearn train_test_split: that shuffles rows randomly.
# For time series data, shuffling means training on 2022 and testing on 2018 —
# that inflates accuracy because the model has seen 'future' data during training.
# Always split time series by date.
TRAIN_END_YEAR = 2020
TEST_START_YEAR = 2021

train_df = df_taska[df_taska['year'] <= TRAIN_END_YEAR].copy()
test_df  = df_taska[df_taska['year'] >= TEST_START_YEAR].copy()

X_train = train_df[TASKA_FEATURES].values
y_train = train_df[TASKA_TARGET].values
X_test  = test_df[TASKA_FEATURES].values
y_test  = test_df[TASKA_TARGET].values

print(f"Train set: {len(train_df):,} rows  (years {train_df['year'].min()}–{train_df['year'].max()})")
print(f"Test set : {len(test_df):,} rows   (years {test_df['year'].min()}–{test_df['year'].max()})")
print()

# ── Scale features ────────────────────────────────────────────────────────────
# Fit scaler on TRAIN only. Apply to both train and test.
# WHY fit on train only: if we fit on all data, the scaler uses information
# from the test set (its mean and std) — that is data leakage.
scaler_a = StandardScaler()
X_train  = scaler_a.fit_transform(X_train)
X_test   = scaler_a.transform(X_test)

print("Train/test split complete. Scaler fitted on training data only.")
print(f"PM2.5_next target range — train: {y_train.min():.1f} to {y_train.max():.1f} µg/m³")
print(f"PM2.5_next target range — test : {y_test.min():.1f} to {y_test.max():.1f} µg/m³")

Train set: 1,203 rows  (years 2015–2020)
Test set : 1,202 rows   (years 2021–2023)

Train/test split complete. Scaler fitted on training data only.
PM2.5_next target range — train: 6.5 to 247.7 µg/m³
PM2.5_next target range — test : 7.9 to 263.9 µg/m³


---
## Cell 9 — Task A: Linear Regression Baseline

Linear Regression is always the first model.  
If test R² ≥ 0.6, we stop here — no need for a more complex model.

In [9]:
print("=" * 55)
print("TASK A — LINEAR REGRESSION (baseline)")
print("=" * 55)
print()

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_train_lr = lr_model.predict(X_train)
y_pred_test_lr  = lr_model.predict(X_test)

lr_train_r2  = r2_score(y_train, y_pred_train_lr)
lr_test_r2   = r2_score(y_test,  y_pred_test_lr)
lr_test_mae  = mean_absolute_error(y_test, y_pred_test_lr)

print(f"Train R²  : {lr_train_r2:.4f}")
print(f"Test  R²  : {lr_test_r2:.4f}")
print(f"Test  MAE : {lr_test_mae:.2f} µg/m³")
print()
print("Interpretation:")
print(f"   R² = {lr_test_r2:.3f} means the model explains {lr_test_r2*100:.1f}% of the variance")
print(f"   MAE = {lr_test_mae:.2f} µg/m³ means predictions are off by {lr_test_mae:.2f} µg/m³ on average")
print()

# ── Print feature coefficients ────────────────────────────────────────────────
# WHY: Coefficient signs tell us the direction of each feature's effect.
# Negative WS coefficient = more wind → lower PM2.5 (physically correct: wind disperses pollution)
# Negative RF coefficient = more rain → lower PM2.5 (wet deposition)
print("Feature coefficients (after scaling — larger absolute value = more influence):")
for feat, coef in sorted(zip(TASKA_FEATURES, lr_model.coef_),
                         key=lambda x: abs(x[1]), reverse=True):
    direction = '+' if coef > 0 else '-'
    print(f"   {feat:25s}: {direction}{abs(coef):.4f}")

print()

# ── Decision: proceed to Random Forest? ──────────────────────────────────────
if lr_test_r2 >= 0.6:
    print(f"Test R² = {lr_test_r2:.3f} >= 0.6 — Linear Regression is sufficient.")
    print("Random Forest is optional but not required for a strong result.")
    PROCEED_TO_RF = False
else:
    print(f"Test R² = {lr_test_r2:.3f} < 0.6 — proceeding to Random Forest in Cell 10.")
    PROCEED_TO_RF = True

TASK A — LINEAR REGRESSION (baseline)

Train R²  : 0.6280
Test  R²  : 0.4775
Test  MAE : 19.60 µg/m³

Interpretation:
   R² = 0.478 means the model explains 47.8% of the variance
   MAE = 19.60 µg/m³ means predictions are off by 19.60 µg/m³ on average

Feature coefficients (after scaling — larger absolute value = more influence):
   PM25_lag1                : +35.3103
   month_sin                : -18.3107
   month_cos                : +12.8677
   PM25_lag2                : -11.7001
   RH (%)_mean              : -10.0019
   WS (m/s)_mean            : -4.0847
   AT (degree C)_mean       : +3.0810
   RF (mm)_mean             : -2.5777
   year                     : +1.0411

Test R² = 0.478 < 0.6 — proceeding to Random Forest in Cell 10.


---
## Cell 10 — Task A: Random Forest

Random Forest is a collection of many decision trees trained on random subsets of the data.  
It handles non-linear relationships (like the sharp winter PM2.5 spike) better than linear regression.

**If Cell 9 printed "Linear Regression is sufficient", you can skip this cell.**  
Running it anyway is fine — it will just confirm that Random Forest also performs well.

In [10]:
print("=" * 55)
print("TASK A — RANDOM FOREST")
print("=" * 55)
print()

# ── Fit Random Forest ─────────────────────────────────────────────────────────
# n_estimators=100: 100 trees in the forest — standard starting point
# max_depth=10: limits tree depth to prevent overfitting
# random_state=42: makes results reproducible — same seed = same result every run
# n_jobs=-1: use all available CPU cores to speed up training
rf_model = RandomForestRegressor(
    n_estimators = 100,
    max_depth    = 10,
    random_state = 42,
    n_jobs       = -1,
)
rf_model.fit(X_train, y_train)

y_pred_train_rf = rf_model.predict(X_train)
y_pred_test_rf  = rf_model.predict(X_test)

rf_train_r2  = r2_score(y_train, y_pred_train_rf)
rf_test_r2   = r2_score(y_test,  y_pred_test_rf)
rf_test_mae  = mean_absolute_error(y_test, y_pred_test_rf)

print(f"Train R²  : {rf_train_r2:.4f}")
print(f"Test  R²  : {rf_test_r2:.4f}")
print(f"Test  MAE : {rf_test_mae:.2f} µg/m³")
print()

# ── Compare with Linear Regression ───────────────────────────────────────────
print("Model comparison:")
print(f"   Linear Regression  — Test R²: {lr_test_r2:.4f}  MAE: {lr_test_mae:.2f} µg/m³")
print(f"   Random Forest      — Test R²: {rf_test_r2:.4f}  MAE: {rf_test_mae:.2f} µg/m³")
print()

# ── Overfitting check ─────────────────────────────────────────────────────────
# WHY: If train R² is much higher than test R², the model memorised the training
# data but can't generalise. This is called overfitting.
gap = rf_train_r2 - rf_test_r2
if gap > 0.2:
    print(f"WARNING: Train R² ({rf_train_r2:.3f}) is {gap:.3f} higher than Test R² ({rf_test_r2:.3f})")
    print("This suggests overfitting. Consider reducing max_depth or n_estimators.")
else:
    print(f"Overfitting check: gap = {gap:.3f} — acceptable (below 0.2 threshold)")

# ── Feature importance ────────────────────────────────────────────────────────
print()
print("Feature importances (% contribution to the model's decisions):")
importances = rf_model.feature_importances_
for feat, imp in sorted(zip(TASKA_FEATURES, importances),
                        key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 50)
    print(f"   {feat:25s}: {imp*100:5.1f}%  {bar}")

TASK A — RANDOM FOREST

Train R²  : 0.9490
Test  R²  : 0.5943
Test  MAE : 16.04 µg/m³

Model comparison:
   Linear Regression  — Test R²: 0.4775  MAE: 19.60 µg/m³
   Random Forest      — Test R²: 0.5943  MAE: 16.04 µg/m³

This suggests overfitting. Consider reducing max_depth or n_estimators.

Feature importances (% contribution to the model's decisions):
   PM25_lag1                :  37.6%  ██████████████████
   month_cos                :  23.0%  ███████████
   month_sin                :  13.0%  ██████
   PM25_lag2                :   7.2%  ███
   AT (degree C)_mean       :   5.1%  ██
   WS (m/s)_mean            :   5.1%  ██
   RH (%)_mean              :   4.4%  ██
   RF (mm)_mean             :   2.7%  █
   year                     :   1.8%  


---
## Cell 11 — Save Task A Results and Predictions

In [11]:
# ── Choose which model's predictions to save ──────────────────────────────────
# WHY: We save the better model's predictions. If Random Forest test R² is
# meaningfully better (> 0.05 improvement), use Random Forest. Otherwise Linear.
use_rf = rf_test_r2 > (lr_test_r2 + 0.05)

if use_rf:
    best_model_name  = 'Random Forest'
    best_test_r2     = rf_test_r2
    best_test_mae    = rf_test_mae
    y_pred_best_test = y_pred_test_rf
else:
    best_model_name  = 'Linear Regression'
    best_test_r2     = lr_test_r2
    best_test_mae    = lr_test_mae
    y_pred_best_test = y_pred_test_lr

print(f"Best model selected: {best_model_name}")
print(f"   Test R²  : {best_test_r2:.4f}")
print(f"   Test MAE : {best_test_mae:.2f} µg/m³")
print()

# ── Save predictions vs actuals for test set ──────────────────────────────────
# WHY: Saving predictions lets you create charts in a report showing
# predicted vs actual PM2.5 values city by city.
test_predictions = test_df[['state', 'city', 'year', 'month',
                             'PM2.5 (ug/m3)_mean', TASKA_TARGET]].copy()
test_predictions['PM25_predicted'] = y_pred_best_test
test_predictions['error']          = (test_predictions['PM25_predicted']
                                      - test_predictions[TASKA_TARGET]).round(2)
test_predictions['model_used']     = best_model_name

pred_path = OUTPUT_DIR / 'taska_predictions.csv'
test_predictions.to_csv(pred_path, index=False)
print(f"Test predictions saved: {pred_path}")
print(f"   {len(test_predictions):,} rows — one per city-month in test period")
print()

# ── Save model summary ────────────────────────────────────────────────────────
model_summary = pd.DataFrame([{
    'task'           : 'Task A',
    'model'          : best_model_name,
    'target'         : TASKA_TARGET,
    'features'       : str(TASKA_FEATURES),
    'train_years'    : f'up to {TRAIN_END_YEAR}',
    'test_years'     : f'{TEST_START_YEAR}+',
    'n_train'        : len(train_df),
    'n_test'         : len(test_df),
    'lr_test_r2'     : round(lr_test_r2, 4),
    'rf_test_r2'     : round(rf_test_r2, 4),
    'best_test_r2'   : round(best_test_r2, 4),
    'best_test_mae'  : round(best_test_mae, 2),
}])

summary_path = OUTPUT_DIR / 'phase5_model_summary.csv'
model_summary.to_csv(summary_path, index=False)
print(f"Model summary saved: {summary_path}")

Best model selected: Random Forest
   Test R²  : 0.5943
   Test MAE : 16.04 µg/m³

Test predictions saved: ..\outputs\taska_predictions.csv
   1,202 rows — one per city-month in test period

Model summary saved: ..\outputs\phase5_model_summary.csv


---
## Cell 12 — Phase 5 Complete Summary

In [12]:
print("=" * 60)
print("PHASE 5 COMPLETE — SUMMARY")
print("=" * 60)
print()

print("FILES SAVED:")
print(f"  outputs/taskb_correlation_results.csv")
print(f"      Pearson r + p-value for PM2.5 vs each of the 21 disease causes")
print(f"  outputs/taskb_regression_results.csv")
print(f"      Linear regression R² and MAE per disease cause")
print(f"  outputs/taska_predictions.csv")
print(f"      Predicted vs actual PM2.5 for test years 2021-2023")
print(f"  outputs/phase5_model_summary.csv")
print(f"      One-row summary of best model and its performance metrics")
print()

print("TASK B RESULTS:")
n_sig_corr = sum(r['significant_05'] for r in taskb_corr_results)
print(f"  {n_sig_corr} of {len(taskb_corr_results)} disease causes show"
      f" statistically significant correlation with PM2.5 (p < 0.05)")
if taskb_corr_results:
    best_corr = max(taskb_corr_results, key=lambda x: x['pearson_r'])
    print(f"  Strongest correlation: '{best_corr['cause_name']}'")
    print(f"    r = {best_corr['pearson_r']:.3f}, p = {best_corr['p_value']:.4f}")
print()

print("TASK A RESULTS:")
print(f"  Best model   : {best_model_name}")
print(f"  Test R²      : {best_test_r2:.4f} (explains {best_test_r2*100:.1f}% of variance)")
print(f"  Test MAE     : {best_test_mae:.2f} µg/m³ average prediction error")
print(f"  Train period : up to {TRAIN_END_YEAR}")
print(f"  Test period  : {TEST_START_YEAR}–2023")
print()

print("HOW TO CITE THESE RESULTS IN THE REPORT:")
print("  Task B: 'A Pearson correlation analysis across [n] state-year pairs")
print("  revealed a statistically significant positive correlation between")
print("  annual mean PM2.5 and [cause] death rate (r = X, p = Y).'")
print()
print("  Task A: 'A [Linear Regression / Random Forest] model trained on")
print(f"  monthly city-level data from 2015-{TRAIN_END_YEAR} predicted next-month PM2.5")
print(f"  with R² = {best_test_r2:.3f} and MAE = {best_test_mae:.2f} µg/m³ on the 2021-2023 test set.'")
print()

print("LIMITATIONS TO STATE IN THE REPORT:")
print("  1. Task B uses state-annual averages — cannot prove causation,")
print("     only correlation. Many confounding factors exist (income,")
print("     healthcare access, smoking rates, age distribution).")
print("  2. Task A uses monthly averages — model cannot predict daily peaks.")
print("  3. Most cities are tier_3/tier_4 (< 5 years data) — excluded from")
print("     Task A. Results represent well-monitored cities only.")
print("  4. Disease data is from IHME GBD estimates, not actual death counts —")
print("     GBD uses modelling to fill gaps in India's civil registration data.")
print("=" * 60)

PHASE 5 COMPLETE — SUMMARY

FILES SAVED:
  outputs/taskb_correlation_results.csv
      Pearson r + p-value for PM2.5 vs each of the 21 disease causes
  outputs/taskb_regression_results.csv
      Linear regression R² and MAE per disease cause
  outputs/taska_predictions.csv
      Predicted vs actual PM2.5 for test years 2021-2023
  outputs/phase5_model_summary.csv
      One-row summary of best model and its performance metrics

TASK B RESULTS:
  17 of 21 disease causes show statistically significant correlation with PM2.5 (p < 0.05)
  Strongest correlation: 'Upper respiratory infections'
    r = 0.684, p = 0.0000

TASK A RESULTS:
  Best model   : Random Forest
  Test R²      : 0.5943 (explains 59.4% of variance)
  Test MAE     : 16.04 µg/m³ average prediction error
  Train period : up to 2020
  Test period  : 2021–2023

HOW TO CITE THESE RESULTS IN THE REPORT:
  Task B: 'A Pearson correlation analysis across [n] state-year pairs
  revealed a statistically significant positive correlatio